# 03 — KPIs marketing & stratégie segment × canal

**Modules couverts : M5 (KPIs des campagnes) et M7 (stratégie de ciblage par canal)**  
Responsable : Pascal

Ce notebook calcule les KPI des campagnes marketing, puis construit une stratégie
segment × canal à partir du dataset officiel de segmentation clients.

**Entrées :**
- `data/raw/marketing_data.csv` : campagnes marketing ;
- `data/raw/sales_data.csv` : ventes utilisées pour calculer l'AOV ;
- `data/processed/dataset_clean.csv` : données clients/ventes nettoyées ;
- `segments_clients*.csv` : dataset officiel de segmentation, recherché sous la racine du projet.

**Sorties :** `data/processed/marketing_kpis.csv` et `data/processed/strategie_segments.csv`.

**Hypothèses et limites :**
1. Les campagnes ne contiennent pas de revenu réel par campagne. Le revenu estimé est donc `Conversions × AOV`, où l'AOV est calculé depuis `sales_data.csv`.
2. Les ventes actuelles contiennent cinq lignes et cinq `Sale_ID` uniques. L'AOV est calculé comme le revenu total des lignes (`Quantity × Sale_Price`) divisé par le nombre de ventes.
3. Le dataset officiel détecté contient une colonne `cluster` avec les valeurs `0`, `1`, `2` et `3`, sans profil métier associé. Ces clusters sont conservés tels quels ; aucune correspondance fictive avec Premium, Standard ou Economique n'est créée.
4. Les campagnes ne fournissent pas de KPI par segment client. Pour un cluster sans règle métier nommée, le canal recommandé repose donc sur le meilleur ROI global, calculé dynamiquement.
5. Les divisions par zéro produisent une valeur manquante contrôlée, jamais une valeur infinie.

In [9]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)

## 0. Localisation des dossiers

In [10]:
def trouver_base_dir() -> Path:
    origines = []
    if "__file__" in globals():
        origines.append(Path(__file__).resolve().parent)
    origines.append(Path.cwd().resolve())

    for origine in origines:
        candidats = [origine, *origine.parents]
        for candidat in candidats:
            if (candidat / "data" / "raw" / "marketing_data.csv").exists():
                return candidat
            projets_enfants = candidat.glob("*/data/raw/marketing_data.csv")
            for fichier_marketing in projets_enfants:
                return fichier_marketing.parents[2]

    raise FileNotFoundError(
        "Impossible de trouver la racine du projet contenant data/raw/marketing_data.csv."
    )


BASE_DIR = trouver_base_dir()
DATA_RAW = BASE_DIR / "data" / "raw"
DATA_PROCESSED = BASE_DIR / "data" / "processed"
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

print(f"Racine du projet       : {BASE_DIR}")
print(f"Dossier de donnees     : {DATA_RAW}")
print(f"Dossier de sortie      : {DATA_PROCESSED}")

Racine du projet       : /media/Celesy20/M1 OCC/ML & DL/Examen M1/Segmentation-Client-Marketing-IA
Dossier de donnees     : /media/Celesy20/M1 OCC/ML & DL/Examen M1/Segmentation-Client-Marketing-IA/data/raw
Dossier de sortie      : /media/Celesy20/M1 OCC/ML & DL/Examen M1/Segmentation-Client-Marketing-IA/data/processed


## 1. Chargement et validation des données

`dataset_clean.csv` est cherché dans `data/processed/` (sortie M2), avec repli sur `data/raw/` s'il n'y est pas encore.

In [11]:
COLONNES_MARKETING_ATTENDUES = {
    "Campaign_ID", "Channel", "Budget", "Impressions", "Clicks", "Conversions"
}
COLONNES_SALES_ATTENDUES = {"Sale_ID", "Quantity", "Sale_Price"}
COLONNES_DATASET_CLEAN_ATTENDUES = {"Customer_ID", "Quantity", "Sale_Price", "Total_Spent"}


def valider_colonnes(df: pd.DataFrame, attendues: set[str], nom_fichier: str) -> None:
    manquantes = sorted(attendues - set(df.columns))
    if manquantes:
        raise ValueError(
            f"{nom_fichier} : colonnes manquantes {manquantes}.\n"
            f"Colonnes disponibles : {list(df.columns)}"
        )
    if df.empty:
        raise ValueError(f"{nom_fichier} est vide.")


def charger_csv(chemin: Path, colonnes_attendues: set[str]) -> pd.DataFrame:
    if not chemin.exists():
        raise FileNotFoundError(f"Fichier introuvable : {chemin}")
    df = pd.read_csv(chemin)
    valider_colonnes(df, colonnes_attendues, chemin.name)
    return df


marketing = charger_csv(DATA_RAW / "marketing_data.csv", COLONNES_MARKETING_ATTENDUES)
sales = charger_csv(DATA_RAW / "sales_data.csv", COLONNES_SALES_ATTENDUES)
dataset_clean = charger_csv(
    DATA_PROCESSED / "dataset_data_final.csv",
    COLONNES_DATASET_CLEAN_ATTENDUES,
)

print(f"Campagnes marketing : {len(marketing)}")
print(f"Ventes              : {len(sales)}")
print(
    f"Lignes dataset_clean : {len(dataset_clean)} | "
    f"Clients uniques : {dataset_clean['Customer_ID'].nunique()}"
)
dataset_clean.head()

Campagnes marketing : 5
Ventes              : 5
Lignes dataset_clean : 5 | Clients uniques : 4


,Sale_ID,Product_ID,Customer_ID,Date,Quantity,Sale_Price,Channel,Product_Name,Category,Price,Brand,Name,Age,Gender,Location,Join_Date,Total_Spent,Revenue
0,1,101,2001,2023-01-15,2,50.0,Online,T-shirt,Clothing,25.0,Brand A,Alice,28,Female,New York,2022-05-10,500.0,100.0
1,2,102,2002,2023-01-16,1,75.0,In-Store,Jeans,Clothing,75.0,Brand B,Bob,35,Male,Los Angeles,2022-06-15,750.0,75.0
2,3,103,2001,2023-01-17,3,30.0,Online,Sneakers,Footwear,30.0,Brand C,Alice,28,Female,New York,2022-05-10,500.0,90.0
3,4,104,2003,2023-01-18,1,120.0,In-Store,Jacket,Outerwear,120.0,Brand D,Charlie,22,Male,Chicago,2022-07-20,300.0,120.0
4,5,105,2004,2023-01-19,2,45.0,Online,Hat,Accessories,22.5,Brand E,Diana,30,Female,Houston,2022-08-25,600.0,90.0


## 2. KPIs par campagne

CTR, taux de conversion, CPC, CPA calculés directement sur `marketing_data.csv`. Le revenu et le ROI sont **estimés** via le panier moyen observé dans `dataset_clean.csv` (voir hypothèse 1 en introduction).

In [12]:
COLONNES_NUMERIQUES_MARKETING = [
    "Budget", "Impressions", "Clicks", "Conversions"
]


def verifier_colonnes_numeriques(df: pd.DataFrame, colonnes: list[str], nom_fichier: str) -> None:
    valeurs_invalides = {}
    for colonne in colonnes:
        valeurs = pd.to_numeric(df[colonne], errors="coerce")
        if valeurs.isna().any():
            valeurs_invalides[colonne] = int(valeurs.isna().sum())
    if valeurs_invalides:
        raise ValueError(
            f"{nom_fichier} contient des valeurs numeriques invalides : {valeurs_invalides}"
        )


def calculer_panier_moyen(sales: pd.DataFrame) -> float:
    verifier_colonnes_numeriques(sales, ["Quantity", "Sale_Price"], "sales_data.csv")
    ventes = sales["Quantity"] * sales["Sale_Price"]
    nombre_ventes = sales["Sale_ID"].nunique()
    if nombre_ventes == 0:
        raise ValueError("sales_data.csv ne contient aucune vente exploitable.")
    return float(ventes.sum() / nombre_ventes)


def division_sure(numerateur: pd.Series, denominateur: pd.Series) -> pd.Series:
    denominateur_non_nul = denominateur.where(denominateur != 0)
    return numerateur.div(denominateur_non_nul)


def calculer_kpis_campagnes(marketing: pd.DataFrame, aov: float) -> pd.DataFrame:
    verifier_colonnes_numeriques(
        marketing,
        COLONNES_NUMERIQUES_MARKETING,
        "marketing_data.csv",
    )
    resultats = marketing.copy()

    resultats["CTR"] = (
        division_sure(resultats["Clicks"], resultats["Impressions"]) * 100
    ).round(2)
    resultats["Conversion_Rate"] = (
        division_sure(resultats["Conversions"], resultats["Clicks"]) * 100
    ).round(2)
    resultats["CPC"] = division_sure(
        resultats["Budget"], resultats["Clicks"]
    ).round(2)
    resultats["CPA"] = division_sure(
        resultats["Budget"], resultats["Conversions"]
    ).round(2)
    resultats["Revenue_Estimated"] = (resultats["Conversions"] * aov).round(2)
    resultats["ROI_Percent"] = (
        division_sure(
            resultats["Revenue_Estimated"] - resultats["Budget"],
            resultats["Budget"],
        )
        * 100
    ).round(1)

    colonnes_sortie = [
        "Campaign_ID", 
        "Channel", 
        "Budget", 
        "Impressions", 
        "Clicks",
        "Conversions", 
        "CTR", 
        "Conversion_Rate", 
        "CPC",
        "CPA", 
        "Revenue_Estimated", 
        "ROI_Percent",
    ]
    sortie = resultats[colonnes_sortie]
    colonnes_kpi = [colonne for colonne in colonnes_sortie if colonne.endswith("%)") or "EUR" in colonne]
    if sortie[colonnes_kpi].replace([float("inf"), float("-inf")], pd.NA).isna().all().any():
        raise ValueError("Un KPI est entierement indetermine : verifiez les denominateurs.")
    return sortie


aov = calculer_panier_moyen(sales)
print(f"Panier moyen estime (AOV) : {aov:.2f} EUR")

kpis = calculer_kpis_campagnes(marketing, aov)
kpis

Panier moyen estime (AOV) : 95.00 EUR


,Campaign_ID,Channel,Budget,Impressions,Clicks,Conversions,CTR,Conversion_Rate,CPC,CPA,Revenue_Estimated,ROI_Percent
0,1,Online,1000.0,50000,2000,150,4.00,7.50,0.50,6.67,14250.0,1325.0
1,2,In-Store,1500.0,30000,500,100,1.67,20.00,3.00,15.00,9500.0,533.3
2,3,Social,2000.0,40000,1500,200,3.75,13.33,1.33,10.00,19000.0,850.0
3,4,Email,500.0,20000,1000,50,5.00,5.00,0.50,10.00,4750.0,850.0
4,5,TV,3000.0,60000,3000,250,5.00,8.33,1.00,12.00,23750.0,691.7


In [13]:
kpis.to_csv(DATA_PROCESSED / "marketing_kpis.csv", index=False)
print(f"Ecrit : {(DATA_PROCESSED / 'marketing_kpis.csv').resolve()}")

Ecrit : /media/Celesy20/M1 OCC/ML & DL/Examen M1/Segmentation-Client-Marketing-IA/data/processed/marketing_kpis.csv


## 3. Partie M7 — Stratégie segment × canal

Le notebook recherche automatiquement le fichier `segments_clients*.csv` sous la racine
du projet. Il utilise la colonne `Segment` si elle existe, sinon normalise la colonne
réelle `cluster` ou `Cluster` en `Segment`, sans créer de segmentation provisoire.

Les clusters numériques du dataset actuel sont conservés tels quels. Comme aucun profil
métier n'est fourni pour ces clusters, la stratégie applique le meilleur ROI global comme
règle de repli et le signale dans la colonne `Profil`.

In [14]:
def trouver_dataset_segmentation(base_dir: Path) -> Path:
    candidats = sorted(
        chemin
        for chemin in base_dir.rglob("segments_clients*.csv")
        if chemin.is_file()
    )
    if not candidats:
        raise FileNotFoundError(
            f"Aucun dataset de segmentation segments_clients*.csv trouve sous {base_dir}."
        )
    if len(candidats) > 1:
        print(f"[!] Plusieurs datasets trouves, utilisation de : {candidats[0]}")
    return candidats[0]


def charger_segments_clients(base_dir: Path) -> pd.DataFrame:
    chemin = trouver_dataset_segmentation(base_dir)
    segments = pd.read_csv(chemin)
    valider_colonnes(segments, {"Customer_ID", "Total_Spent"}, chemin.name)

    if "Segment" in segments.columns:
        nom_segment = "Segment"
    elif "cluster" in segments.columns:
        nom_segment = "cluster"
    elif "Cluster" in segments.columns:
        nom_segment = "Cluster"
    else:
        raise ValueError(
            f"{chemin.name} : colonne de segmentation absente. "
            f"Colonnes disponibles : {list(segments.columns)}"
        )

    segments = segments.rename(columns={nom_segment: "Segment"})
    segments["Segment"] = segments["Segment"].astype(str)
    if segments["Segment"].str.strip().eq("").any():
        raise ValueError(f"{chemin.name} contient des segments vides.")

    print(f"[OK] Dataset de segmentation utilise : {chemin}")
    print(f"[OK] Segments reels : {sorted(segments['Segment'].unique())}")
    return segments


customers_segmentes = charger_segments_clients(BASE_DIR)
customers_segmentes

[OK] Dataset de segmentation utilise : /media/Celesy20/M1 OCC/ML & DL/Examen M1/Segmentation-Client-Marketing-IA/data/processed/segments_clients.csv
[OK] Segments reels : ['0', '1']


,Customer_ID,Name,Age,Gender,Location,Join_Date,Total_Spent,last_purchase_date,purchase_frequency,monetary_total,total_quantity,favorite_category,favorite_brand,preferred_channel,nb_distinct_categories,recency_days,avg_basket_value,avg_quantity_per_purchase,tenure_days,Segment
0,2001,Alice,28,Female,New York,2022-05-10,500.0,2023-01-17,2,190.0,5,Clothing,Brand A,Online,2,3,95.0,2.5,255,1
1,2002,Bob,35,Male,Los Angeles,2022-06-15,750.0,2023-01-16,1,75.0,1,Clothing,Brand B,In-Store,1,4,75.0,1.0,219,0
2,2003,Charlie,22,Male,Chicago,2022-07-20,300.0,2023-01-18,1,120.0,1,Outerwear,Brand D,In-Store,1,2,120.0,1.0,184,0
3,2004,Diana,30,Female,Houston,2022-08-25,600.0,2023-01-19,1,90.0,2,Accessories,Brand E,Online,1,1,90.0,2.0,148,0


In [15]:
REGLES_SEGMENTS = {
    "Premium": ("ROI_Percent", "max"),
    "Standard": ("CPA", "min"),
    "Economique": ("Conversion_Rate", "max"),
}

PROFILS_SEGMENTS = {
    "Premium": "Clients a forte valeur, cible prioritaire de fidelisation/upsell",
    "Standard": "Clients a valeur moyenne, a faire progresser vers Premium",
    "Economique": "Clients sensibles au prix, besoin d'un declencheur d'achat direct",
}

REGLE_CLUSTER_SANS_PROFIL = ("ROI_Percent", "max")
PROFIL_CLUSTER_SANS_PROFIL = (
    "Cluster officiel issu de la segmentation clients "
    "(profil metier non fourni dans le dataset)"
)


def choisir_campagne(kpis: pd.DataFrame, metrique: str, mode: str) -> pd.Series:
    if metrique not in kpis.columns:
        raise ValueError(f"KPI requis absent : {metrique}")
    campagnes_valides = kpis.dropna(subset=[metrique])
    if campagnes_valides.empty:
        raise ValueError(f"Aucune valeur exploitable pour le KPI : {metrique}")
    indice = (
        campagnes_valides[metrique].idxmax()
        if mode == "max"
        else campagnes_valides[metrique].idxmin()
    )
    return campagnes_valides.loc[indice]


def construire_strategie_segments(
    customers_segmentes: pd.DataFrame,
    kpis: pd.DataFrame,
) -> pd.DataFrame:

    # ---------------------------------------------------------
    # 1. Vérification des colonnes obligatoires
    # ---------------------------------------------------------
    colonnes_obligatoires = {
        "Customer_ID",
        "Total_Spent",
        "Segment",
        "Age",
        "purchase_frequency",
        "recency_days",
    }

    valider_colonnes(
        customers_segmentes,
        colonnes_obligatoires,
        "dataset de segmentation clients",
    )

    if kpis.empty:
        raise ValueError("Le tableau des KPI est vide.")

    # ---------------------------------------------------------
    # 2. Conversion numérique
    # ---------------------------------------------------------
    customers_segmentes = customers_segmentes.copy()

    colonnes_numeriques = [
        "Age",
        "Total_Spent",
        "purchase_frequency",
        "recency_days",
    ]

    for colonne in colonnes_numeriques:
        customers_segmentes[colonne] = pd.to_numeric(
            customers_segmentes[colonne],
            errors="coerce"
        )

    # ---------------------------------------------------------
    # 3. Nettoyage du segment / cluster
    # ---------------------------------------------------------
    customers_segmentes["Segment"] = (
        customers_segmentes["Segment"]
        .astype(str)
        .str.strip()
    )

    # ---------------------------------------------------------
    # 4. Classement des campagnes par ROI
    # ---------------------------------------------------------
    kpis_valides = kpis.dropna(
        subset=["ROI_Percent", "CPA", "Channel"]
    ).copy()

    if kpis_valides.empty:
        raise ValueError(
            "Aucune campagne exploitable pour calculer les canaux."
        )

    # Canal prioritaire = meilleur ROI
    campagnes_roi = kpis_valides.sort_values(
        "ROI_Percent",
        ascending=False
    ).reset_index(drop=True)

    campagne_prioritaire = campagnes_roi.iloc[0]

    canal_prioritaire = campagne_prioritaire["Channel"]
    roi_prioritaire = float(campagne_prioritaire["ROI_Percent"])
    cpa_prioritaire = float(campagne_prioritaire["CPA"])

    # Canal secondaire = deuxième meilleur ROI
    if len(campagnes_roi) >= 2:
        campagne_secondaire = campagnes_roi.iloc[1]
        canal_secondaire = campagne_secondaire["Channel"]
    else:
        canal_secondaire = canal_prioritaire

    # ---------------------------------------------------------
    # 5. Agrégation des données par cluster
    # ---------------------------------------------------------
    resume = (
        customers_segmentes
        .groupby("Segment", dropna=False)
        .agg(
            Nb_clients=("Customer_ID", "nunique"),
            Age_moyen=("Age", "mean"),
            Montant_moyen=("Total_Spent", "mean"),
            Frequence_moyenne=("purchase_frequency", "mean"),
            Recence_moyenne_jours=("recency_days", "mean"),
        )
        .reset_index()
    )

    # ---------------------------------------------------------
    # 6. Création des colonnes demandées
    # ---------------------------------------------------------
    resultats = []

    for _, ligne in resume.iterrows():

        cluster = str(ligne["Segment"])

        # -----------------------------------------------------
        # Persona
        # -----------------------------------------------------
        # Le PDF précise qu'aucun profil métier n'est fourni.
        # On conserve donc le cluster réel sans inventer
        # Premium / Standard / Economique.
        persona = f"Cluster {cluster}"

        # -----------------------------------------------------
        # Message recommandé
        # -----------------------------------------------------
        message = (
            f"Communication personnalisée pour le cluster {cluster}, "
            f"avec mise en avant des offres adaptées au comportement "
            f"d'achat des clients."
        )

        # -----------------------------------------------------
        # Part du budget
        # -----------------------------------------------------
        # Règle simple : canal prioritaire majoritaire.
        # Cette valeur est une règle métier configurable.
        part_budget = 70.0

        resultats.append({
            "Persona": persona,
            "Nb_clients": int(ligne["Nb_clients"]),
            "Age_moyen": round(float(ligne["Age_moyen"]), 2),
            "Montant_moyen": round(float(ligne["Montant_moyen"]), 2),
            "Frequence_moyenne": round(
                float(ligne["Frequence_moyenne"]), 2
            ),
            "Recence_moyenne_jours": round(
                float(ligne["Recence_moyenne_jours"]), 2
            ),
            "Canal_prioritaire": canal_prioritaire,
            "Canal_secondaire": canal_secondaire,
            "ROI_canal_prioritaire_pct": round(
                roi_prioritaire, 2
            ),
            "CPA_canal_prioritaire": round(
                cpa_prioritaire, 2
            ),
            "Message_recommande": message,
            "Part_budget_recommandee_pct": part_budget,
            "cluster": cluster,
        })

    strategie = pd.DataFrame(resultats)

    # ---------------------------------------------------------
    # 7. Ordre exact des colonnes
    # ---------------------------------------------------------
    colonnes_finales = [
        "Persona",
        "Nb_clients",
        "Age_moyen",
        "Montant_moyen",
        "Frequence_moyenne",
        "Recence_moyenne_jours",
        "Canal_prioritaire",
        "Canal_secondaire",
        "ROI_canal_prioritaire_pct",
        "CPA_canal_prioritaire",
        "Message_recommande",
        "Part_budget_recommandee_pct",
        "cluster",
    ]

    return strategie[colonnes_finales]


strategie = construire_strategie_segments(
    customers_segmentes,
    kpis
)

strategie.to_csv(
    DATA_PROCESSED / "strategie_segments.csv",
    index=False
)

print(
    f"Ecrit : {(DATA_PROCESSED / 'strategie_segments.csv').resolve()}"
)

print("\nstrategie_segments.csv")
print(strategie.to_string(index=False))


# ============================================================
# M6 / KPI MARKETING PAR CANAL
# ============================================================

def construire_kpis_par_canal(kpis: pd.DataFrame) -> pd.DataFrame:
    """
    Agrège les KPI des campagnes par canal marketing.

    Entrée :
        kpis : DataFrame contenant les KPI par campagne

    Sortie :
        DataFrame avec une ligne par canal.
    """

    colonnes_requises = [
        "Channel",
        "Budget",
        "Impressions",
        "Clicks",
        "Conversions",
        "CTR",
        "Conversion_Rate",
        "CPC",
        "CPA",
        "ROI_Percent",
    ]

    # --------------------------------------------------------
    # 1. Vérification
    # --------------------------------------------------------
    valider_colonnes(
        kpis,
        set(colonnes_requises),
        "marketing_kpis"
    )

    # --------------------------------------------------------
    # 2. Agrégation par canal
    # --------------------------------------------------------
    kpis_channel = (
        kpis
        .groupby("Channel", dropna=False)
        .agg(
            Total_Budget=("Budget", "sum"),
            Total_Impressions=("Impressions", "sum"),
            Total_Clicks=("Clicks", "sum"),
            Total_Conversions=("Conversions", "sum"),

            Avg_CTR=("CTR", "mean"),
            Avg_Conversion_Rate=("Conversion_Rate", "mean"),
            Avg_CPC=("CPC", "mean"),
            Avg_CPA=("CPA", "mean"),
            Avg_ROI_Percent=("ROI_Percent", "mean"),
        )
        .reset_index()
    )

    # --------------------------------------------------------
    # 3. Arrondir les KPI
    # --------------------------------------------------------
    kpis_channel["Total_Budget"] = (
        kpis_channel["Total_Budget"].round(2)
    )

    kpis_channel["Avg_CTR"] = (
        kpis_channel["Avg_CTR"].round(2)
    )

    kpis_channel["Avg_Conversion_Rate"] = (
        kpis_channel["Avg_Conversion_Rate"].round(2)
    )

    kpis_channel["Avg_CPC"] = (
        kpis_channel["Avg_CPC"].round(2)
    )

    kpis_channel["Avg_CPA"] = (
        kpis_channel["Avg_CPA"].round(2)
    )

    kpis_channel["Avg_ROI_Percent"] = (
        kpis_channel["Avg_ROI_Percent"].round(2)
    )

    # --------------------------------------------------------
    # 4. Ordre final des colonnes
    # --------------------------------------------------------
    colonnes_finales = [
        "Channel",
        "Total_Budget",
        "Total_Impressions",
        "Total_Clicks",
        "Total_Conversions",
        "Avg_CTR",
        "Avg_Conversion_Rate",
        "Avg_CPC",
        "Avg_CPA",
        "Avg_ROI_Percent",
    ]

    return kpis_channel[colonnes_finales]


# ============================================================
# Génération du fichier marketing_kpis_channel.csv
# ============================================================

marketing_kpis_channel = construire_kpis_par_canal(kpis)

marketing_kpis_channel.to_csv(
    DATA_PROCESSED / "marketing_kpis_channel.csv",
    index=False
)

print(
    f"Écrit : "
    f"{(DATA_PROCESSED / 'marketing_kpis_channel.csv').resolve()}"
)

print("\n===== KPI MARKETING PAR CANAL =====")
print(marketing_kpis_channel.to_string(index=False))


Ecrit : /media/Celesy20/M1 OCC/ML & DL/Examen M1/Segmentation-Client-Marketing-IA/data/processed/strategie_segments.csv

strategie_segments.csv
  Persona  Nb_clients  Age_moyen  Montant_moyen  Frequence_moyenne  Recence_moyenne_jours Canal_prioritaire Canal_secondaire  ROI_canal_prioritaire_pct  CPA_canal_prioritaire                                                                                                         Message_recommande  Part_budget_recommandee_pct cluster
Cluster 0           3       29.0          550.0                1.0                   2.33            Online           Social                     1325.0                   6.67 Communication personnalisée pour le cluster 0, avec mise en avant des offres adaptées au comportement d'achat des clients.                         70.0       0
Cluster 1           1       28.0          500.0                2.0                   3.00            Online           Social                     1325.0                   6.67 Communicati

In [16]:
strategie.to_csv(DATA_PROCESSED / "strategie_segments.csv", index=False)
print(f"Ecrit : {(DATA_PROCESSED / 'strategie_segments.csv').resolve()}")

Ecrit : /media/Celesy20/M1 OCC/ML & DL/Examen M1/Segmentation-Client-Marketing-IA/data/processed/strategie_segments.csv


In [17]:
marketing_sortie = pd.read_csv(DATA_PROCESSED / "marketing_kpis.csv")
strategie_sortie = pd.read_csv(DATA_PROCESSED / "strategie_segments.csv")

colonnes_kpi = [
    "CTR (%)", "Taux_Conversion (%)", "CPC (EUR)", "CPA (EUR)",
    "Revenu_Estime (EUR)", "ROI (%)",
]
assert len(marketing_sortie) == len(marketing)

strategie_sortie = pd.read_csv(
    DATA_PROCESSED / "strategie_segments.csv"
)

# Vérification des colonnes
colonnes_attendues_strategie = [
    "Persona",
    "Nb_clients",
    "Age_moyen",
    "Montant_moyen",
    "Frequence_moyenne",
    "Recence_moyenne_jours",
    "Canal_prioritaire",
    "Canal_secondaire",
    "ROI_canal_prioritaire_pct",
    "CPA_canal_prioritaire",
    "Message_recommande",
    "Part_budget_recommandee_pct",
    "cluster",
]

assert set(colonnes_attendues_strategie).issubset(
    set(strategie_sortie.columns)
)

# Les clusters générés doivent être exactement ceux
# du dataset de segmentation
assert set(
    strategie_sortie["cluster"].astype(str)
) == set(
    customers_segmentes["Segment"].astype(str)
)

# Nombre total de clients
assert strategie_sortie["Nb_clients"].sum() == \
       customers_segmentes["Customer_ID"].nunique()

# Canaux valides
assert set(
    strategie_sortie["Canal_prioritaire"]
).issubset(
    set(marketing["Channel"])
)

assert set(
    strategie_sortie["Canal_secondaire"]
).issubset(
    set(marketing["Channel"])
)

print("Validation des sorties terminée")
print(f"AOV : {aov:.2f} EUR")
print("\nmarketing_kpis.csv")
print(marketing_sortie.to_string(index=False))
print("\nstrategie_segments.csv")
print(strategie_sortie.to_string(index=False))

Validation des sorties terminée
AOV : 95.00 EUR

marketing_kpis.csv
 Campaign_ID  Channel  Budget  Impressions  Clicks  Conversions  CTR  Conversion_Rate  CPC   CPA  Revenue_Estimated  ROI_Percent
           1   Online  1000.0        50000    2000          150 4.00             7.50 0.50  6.67            14250.0       1325.0
           2 In-Store  1500.0        30000     500          100 1.67            20.00 3.00 15.00             9500.0        533.3
           3   Social  2000.0        40000    1500          200 3.75            13.33 1.33 10.00            19000.0        850.0
           4    Email   500.0        20000    1000           50 5.00             5.00 0.50 10.00             4750.0        850.0
           5       TV  3000.0        60000    3000          250 5.00             8.33 1.00 12.00            23750.0        691.7

strategie_segments.csv
  Persona  Nb_clients  Age_moyen  Montant_moyen  Frequence_moyenne  Recence_moyenne_jours Canal_prioritaire Canal_secondaire  ROI_cana